<a href="https://colab.research.google.com/github/Csr9802/Agente-simple/blob/main/agente_modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentes Basados en Modelos

## Definición
Los agentes basados en modelos son una evolución de los agentes reflexivos simples que mantienen un estado interno y utilizan modelos para comprender cómo evoluciona el mundo.

## Componentes Principales

1. **Estado Interno**
   - Almacena el historial de percepciones
   - Mantiene una representación del mundo
   - Permite manejar la observabilidad parcial

2. **Modelo de Transición**
```python
modelo_transicion = {
    "estado_actual": estado_previo + accion,
    "prediccion": efecto_acciones
}
```

3. **Modelo del Sensor**
```python
modelo_sensor = {
    "interpretacion": mapeo_percepcion_estado,
    "ruido": incertidumbre_sensores
}
```

## Ventajas
- Maneja entornos parcialmente observables
- Mejor toma de decisiones con información histórica
- Puede predecir evolución del entorno

## Limitaciones
- Mayor complejidad computacional
- Requiere más memoria
- Necesita modelos precisos

## Ejemplo: Conductor Automático
```python
def agente_conductor():
    estado = actualizar_estado(percepcion)
    prediccion = modelo_transicion(estado, accion)
    return seleccionar_mejor_accion(prediccion)
```

## Aplicaciones
- Robots autónomos
- Sistemas de monitoreo
- Navegación
- Control de procesos industriales

## Comparación con Agentes Reflexivos Simples

| Característica | Reflexivo Simple | Basado en Modelo |
|----------------|------------------|------------------|
| Estado | No | Sí |
| Predicción | No | Sí |
| Memoria | No | Sí |
| Complejidad | Baja | Media |

In [ ]:
import requests
from datetime import datetime

# Estado interno persistente
estado = {
   "ultimo_estado": 0,
   "ultima_revision": datetime.now(),
   "errores_consecutivos": 0,
   "historial_respuestas": {}
}

# Modelo de transición - cómo evoluciona el entorno
modelo_transicion = {
   "umbral_error": 3,    # Máximo de errores consecutivos antes de alerta
   "tiempo_maximo": 300  # Tiempo máximo aceptable entre revisiones (5 min)
}

# Modelo del sensor - cómo interpretamos las percepciones
modelo_sensor = {
   "exito": range(200, 300),
   "error_cliente": range(400, 500),
   "error_servidor": range(500, 600)
}

def actualizar_estado(percepcion: int) -> None:
   """Actualiza el estado interno basado en nueva percepción"""
   tiempo_actual = datetime.now()
   estado["historial_respuestas"][tiempo_actual.isoformat()] = percepcion

   if percepcion in modelo_sensor["exito"]:
       estado["errores_consecutivos"] = 0
   else:
       estado["errores_consecutivos"] += 1

   estado["ultimo_estado"] = percepcion
   estado["ultima_revision"] = tiempo_actual

def interpretar_entrada(codigo_estado: int) -> str:
   """Mapea percepciones a estados abstractos usando modelo del sensor"""
   if codigo_estado in modelo_sensor["exito"]:
       return "EXITO"
   elif codigo_estado in modelo_sensor["error_cliente"]:
       return "ERROR_CLIENTE"
   elif codigo_estado in modelo_sensor["error_servidor"]:
       return "ERROR_SERVIDOR"
   return "DESCONOCIDO"

def regla_coincidente(estado_actual: str) -> str:
   """Determina acción basada en estado actual y modelos"""
   if estado["errores_consecutivos"] >= modelo_transicion["umbral_error"]:
       return f"Alerta: Múltiples fallos detectados. Último estado: {estado_actual}"

   if estado_actual == "EXITO":
       tiempo_transcurrido = (datetime.now() - estado["ultima_revision"]).seconds
       if tiempo_transcurrido > modelo_transicion["tiempo_maximo"]:
           return "Advertencia: Servicio responde pero con demora"
       return "Registro: Servicio operando normalmente"

   elif estado_actual == "ERROR_CLIENTE":
       return "Alerta: Problema de configuración del cliente"
   elif estado_actual == "ERROR_SERVIDOR":
       return "Alerta: Error crítico del servidor"
   return "Advertencia: Estado del servicio desconocido"

def agente_basado_modelo(url: str) -> str:
   """Función principal del agente que procesa percepción y retorna acción"""
   try:
       respuesta = requests.get(url)
       codigo_estado = respuesta.status_code

       # Actualiza estado interno basado en percepción
       actualizar_estado(codigo_estado)

       # Interpreta entrada usando modelo del sensor
       estado_actual = interpretar_entrada(codigo_estado)

       # Selecciona acción basada en estado actual y modelos
       accion = regla_coincidente(estado_actual)

       return accion

   except requests.RequestException as e:
       actualizar_estado(0)
       return f"Error: Falló la conexión - {str(e)}"

# Ejemplo de uso
url = "https://mock.httpstatus.io/200"
resultado = agente_basado_modelo(url)
print(resultado)

Registro: Servicio operando normalmente
